# SQUID measurement cycle  *(thin notebook over the `AutoSQUID` package)*

Run on the **bench PC** (real `pyserial` + `nidaqmx` + `RanLabPythonRepo`). All logic lives in `AutoSQUID/`;
this notebook just builds a `Config` (§0 — the cell you edit) and calls `sq.*`.

For each scan interval it collects **`n_trials` clean traces** within **`max_attempts`** acquisitions,
order-indexing files (`DAQ_…_{i}.txt`, failed `…_{i}_{JUMP|SURGE|RAIL|BADBASE}.txt`), resuming from disk,
and stopping the whole sweep if a reset won't hold. Run cells in order; §1 is read-only, **§2 is ACTIVE**.

## §0 — Config  *(the cell you edit)*

In [ ]:
# add the package's PARENT to sys.path so `import AutoSQUID` works from inside the AutoSQUID/ folder,
# and the bench-PC RanLabPythonRepo (two levels up, like the other notebooks) is importable.
import sys
sys.path.insert(0, "..")          # .../automation  -> exposes the `AutoSQUID` package
sys.path.append("../../")         # .../SQUID/... root for RanLabPythonRepo (same as the other notebooks)
import AutoSQUID as sq
print("AutoSQUID loaded:", [n for n in sq.__all__[:6]], "...")

cfg = sq.Config(
    # --- acquisition ---
    scan_interval_s = [100e-6],        # one cycle per entry; e.g. [4e-6, 20e-6, 100e-6, 500e-6]
    n_points        = 10_000_000,      # length of the consecutive run
    temp_label      = "auto",          # "auto" -> read+round MXC temp; or a literal e.g. "14mK"
    n_trials        = 2,               # target CLEAN traces per scan interval
    # --- live jump check ---
    chunk           = 100_000,         # jump-check cadence (does NOT break the consecutive run)
    jump_v          = 0.5,             # baseline-mean slip (V) flagged as a JUMP
    baseline_chunks = 1,
    # --- temperature ---
    temp_every_s    = 30.0,
    temp_channel    = 6,               # MXC mixing-chamber thermometer
    # --- control (SCC serial) ---
    port            = "COM3",          # PCS102DA may stay open on another port to hold S-lock
    # --- DAQ / output ---
    vrange          = 1.0,             # NI AI range (V); +/-1 V matches previous measurements
    max_attempts    = 4,               # max total acquisitions per interval (clean + failed)
    user            = "Shannon",
    date            = "",              # date subfolder, e.g. "Jun-01-2026"
)
cfg.outdir.mkdir(parents=True, exist_ok=True)
print(f"intervals {[f'{t*1e6:g}us' for t in cfg.scan_intervals]} · {cfg.n_trials} clean each "
      f"(<= {cfg.max_attempts} acq) · {cfg.n_points} pts ({cfg.npts_tag}) · out -> {cfg.outdir}")

## §1 — Checks  *(read-only)*: pick the live NI input channel + confirm the temperature backend

In [ ]:
dev, cfg.daq_ai = sq.detect_ai_channel(cfg)     # auto-pick the PCIe-6320 live ai0 (honors cfg.force_*)
print(f"device {dev.name} ({dev.product_type}) · DAQ_AI = {cfg.daq_ai} · control PORT = {cfg.port}")
try:
    T = sq.read_temp(cfg)
    print(f"MXC temperature: {T:.4f} K  (label {sq.format_temp_label(T)})")
except Exception as e:
    print(f"temperature read FAILED ({e}) — logging running? channel {cfg.temp_channel}?")

## §2 — Run the sweep  *(ACTIVE — sends resets, acquires `n_points`, writes files)*

Resolves the temperature label, then runs one cycle per scan interval (stops the whole sweep on a reset
failure). Every acquisition + reset is appended to `experiment_log.txt`; traces + `_temp.csv` go to `outdir`.

In [ ]:
import datetime
cfg.outdir.mkdir(parents=True, exist_ok=True)
print(f"temperature label for this run: {sq.resolve_temp_label(cfg)}")
print(f"measurement START : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
for tau in cfg.scan_intervals:                       # one cycle per scan interval
    if sq.run_cycle(cfg, tau) == "reset_fail":       # reset won't hold (systemic) -> stop the whole sweep
        print("\n*** STOPPED: reset is not working — no more intervals attempted. "
              "Fix it (port/register/lock — see the protocol §G), then re-run §2: it resumes from disk. ***")
        break

## §3 — Plot raw data  *(reads this run's clean traces + temperature back from disk)*

In [ ]:
sq.plot_run(cfg)
# or a manual list: sq.plot_run(cfg, ["DAQ_4us_15mK_10Mpts_1.txt"])